# HiGAN + Generated-Style SupCon
Enable a Kaggle GPU and Internet, then run all cells.

In [ ]:
# Chỉ cần sửa cell này trước mỗi lần chạy.
REPO_URL = 'https://github.com/<owner>/HiGAN-Contrastive.git'
GIT_BRANCH = 'main'
DATASET_ROOT = '/kaggle/input/<dataset-slug>/iam_dataset'
RUN_MODE = 'contrastive'  # baseline hoặc contrastive
RESUME_PATH = ''  # đường dẫn last.pt của phiên trước, hoặc để trống
CONTRASTIVE_START_STEP = 28841  # step đầu tiên có SupCon
CONTRASTIVE_RAMP_STEPS = 28840  # 0/1 = bật ngay lambda_con tối đa
RECOGNIZER_CHECKPOINT = ''  # recognizer độc lập, hoặc để trống
HWD_CHECKPOINT = ''  # official VGG16_class_10400.pth; trống = tự tải và cache
MAX_HOURS = 11.5
EVAL_MAX_SAMPLES = 512  # tăng lên 1000+ cho báo cáo cuối nếu thời gian cho phép


In [ ]:
REPO_DIR = '/kaggle/working/HiGAN-Contrastive'
!if [ -d "{REPO_DIR}/.git" ]; then git -C "{REPO_DIR}" checkout "{GIT_BRANCH}" && git -C "{REPO_DIR}" pull --ff-only origin "{GIT_BRANCH}"; else git clone --branch "{GIT_BRANCH}" "{REPO_URL}" "{REPO_DIR}"; fi
!git -C "{REPO_DIR}" rev-parse HEAD
%cd {REPO_DIR}


In [ ]:
!bash scripts/kaggle_setup.sh "{DATASET_ROOT}"


In [ ]:
CONFIG = f'configs/kaggle_{RUN_MODE}.yaml'
SMOKE_OUTPUT = f'/kaggle/working/smoke/{RUN_MODE}'
!python train.py --config "{CONFIG}" --dataset-root "{DATASET_ROOT}" --manifest-root /kaggle/working/data --output "{SMOKE_OUTPUT}" --smoke-test --contrastive-start-step "{CONTRASTIVE_START_STEP}" --contrastive-ramp-steps "{CONTRASTIVE_RAMP_STEPS}"
from IPython.display import Image, display
display(Image(filename=f'{SMOKE_OUTPUT}/previews/step_00000001.png'))


In [ ]:
!bash scripts/kaggle_run.sh "{DATASET_ROOT}" "{RUN_MODE}" "{RESUME_PATH}" "{RECOGNIZER_CHECKPOINT}" "{MAX_HOURS}" "{HWD_CHECKPOINT}" "{EVAL_MAX_SAMPLES}" "{CONTRASTIVE_START_STEP}" "{CONTRASTIVE_RAMP_STEPS}"
from pathlib import Path
preview_files = sorted((Path('/kaggle/working/outputs') / RUN_MODE / 'previews').glob('*.png'))
if preview_files:
    display(Image(filename=str(preview_files[-1])))
evaluation_path = Path('/kaggle/working/outputs') / RUN_MODE / 'evaluation.json'
if evaluation_path.exists():
    print(evaluation_path.read_text())
print('/kaggle/working/run_bundle.tar.gz')
